In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

col1 = [
    (1, 'A'),
    (1, 'B'),
    (1, 'C'),
    (2, 'D'),
    (3, 'E')
]

df1 = spark.createDataFrame(col1, ['user_id', 'values'])

In [2]:
col2 = [
    (1, 'X'),
    (2, 'Y')
]

df2 = spark.createDataFrame(col2, ['user_id', 'values'])

# Salt the bigdf

In [3]:
from pyspark.sql.functions import rand, concat, col, lit
df1 = df1.withColumn('salt', (rand() * 3).cast('int')) \
                .withColumn('salted_id', concat(col('user_id'), lit('_'), col('salt')))

df1.show()

+-------+------+----+---------+
|user_id|values|salt|salted_id|
+-------+------+----+---------+
|      1|     A|   0|      1_0|
|      1|     B|   1|      1_1|
|      1|     C|   1|      1_1|
|      2|     D|   0|      2_0|
|      3|     E|   0|      3_0|
+-------+------+----+---------+



In [4]:
salts = [0, 1, 2]

df2 = spark.createDataFrame(
    [(row.user_id, row.values, s) for row in df2.collect() for s in salts],
    ['user_id','values', 'salt']
).withColumn('salted_id', concat(col('user_id'), lit('_'), col('salt')))

df2.show()

+-------+------+----+---------+
|user_id|values|salt|salted_id|
+-------+------+----+---------+
|      1|     X|   0|      1_0|
|      1|     X|   1|      1_1|
|      1|     X|   2|      1_2|
|      2|     Y|   0|      2_0|
|      2|     Y|   1|      2_1|
|      2|     Y|   2|      2_2|
+-------+------+----+---------+



In [8]:
from pyspark.sql.functions import broadcast

joined_columns = df1.join(broadcast(df2), df1.salted_id == df2.salted_id, 'left')
joined_columns.drop('salt', 'salted_id').show()

+-------+------+-------+------+
|user_id|values|user_id|values|
+-------+------+-------+------+
|      1|     A|      1|     X|
|      1|     B|      1|     X|
|      1|     C|      1|     X|
|      2|     D|      2|     Y|
|      3|     E|   NULL|  NULL|
+-------+------+-------+------+

